In [1]:
import random

import numpy as np
import pandas as pd
import seaborn as sns

from matplotlib import pyplot as plt
from utils.config import Config
from data_handler import DataHandler
# from classificators.dummy_classifier import DummyClassifier
# from classificators.random_forest_classifier import RandomForestClassifierSK
# from utils.utils import calculate_mcc_multilabel, plot_per_class_confusion

# pd.options.display.float_format = '{:.6f}'.format

In [2]:
config = Config()
print(config.data)
print(config.prep)
# Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# if you use any other libraries that require seeding, set it here as well (e.g., torch.manual_seed(SEED) for PyTorch)
# -> your results should be reproducible across runs with the same seed


val_mccs = []
test_mccs = []
lr_histories_by_fold = {}

# load data
datahandler = DataHandler(config=config)



DataConfig(dataset_file='cps_data_multi_label.pkl', download_url='https://owncloud.fraunhofer.de/index.php/s/gElpu40mbgK7jau/download', sensor_cols=['Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x'], gyro_cols=['Gyro.x', 'Gyro.y', 'Gyro.z'], test_experiment_id=1, validation_experiment_id=2, label_cols=['Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Standing', 'Docking', 'Forks(entering or leaving front)', 'Forks(entering or leaving side)', 'Wrapping', 'Wrapping(preparation)'], superclass_mapping={'Driving(curve)': 'Driving(curve)', 'Driving(straight)': 'Driving(straight)', 'Lifting(lowering)': 'Lifting(lowering)', 'Lifting(raising)': 'Lifting(raising)', 'Wrapping': 'Turntable wrapping', 'Wrapping(preparation)': 'Stationary processes', 'Docking': 'Stationary processes', 'Forks(entering or leaving front)': 'Stationary processes', 'Forks(entering or leaving side)': 'Stationary processes', 'Standing': 'Stationary processes'})
Preprocessing

In [ ]:

# Leave-one-out: EXPERIMENT_ID = 1..4
for fold in range(1, 5):
    val_id = fold + 1 if fold < 4 else 1

    datahandler.config.data.test_experiment_id = fold
    # validation hat to be different from test
    datahandler.config.data.validation_experiment_id = val_id

    train, val, test, target_vals = datahandler.get_data_loaders()
    # print("Full Data Shape")
    # print(train.shape, val.shape, test.shape)
    # print("Target columns:", target_vals)
    # print("Train columns:", train.columns.tolist())
    # print(train.isna().sum())
    # X_train = train.iloc[:, :8]
    # y_train = train.iloc[:, 8:]
    # X_val = val.iloc[:, :8]
    # y_val = val.iloc[:, 8:]
    # X_test = test.iloc[:, :8]
    # y_test = test.iloc[:, 8:]
    # print(X_train['time'].describe())
    # print((X_train[X_train['time']==0]))
    # sns.scatterplot(X_train['time'])
    print(train.columns.tolist())

Starting data preparation...


['time', 'Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x', 'No loading', 'Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Stationary processes', 'Turntable wrapping']
Starting data preparation...
['time', 'Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x', 'No loading', 'Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Stationary processes', 'Turntable wrapping']
Starting data preparation...
['time', 'Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x', 'No loading', 'Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Stationary processes', 'Turntable wrapping']
Starting data preparation...
['time', 'Acc.x', 'Acc.y', 'Acc.z', 'Gyro.x', 'Gyro.y', 'Gyro.z', 'Baro.x', 'No loading', 'Driving(straight)', 'Driving(curve)', 'Lifting(raising)', 'Lifting(lowering)', 'Stationary processes', 'Turntable wrapping']


In [4]:
def plot_sensor_data_over_time(X_train, col):
    plt.figure(figsize=(12, 10))
    plt.plot(X_train['time'], X_train[col], label=col)
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.legend()
    plt.show()


In [5]:
def plot_sensor_correlation(X_train):
    plt.figure(figsize=(12, 10))
    corr = X_train[['Acc.x','Acc.y','Acc.z','Gyro.x','Gyro.y','Gyro.z','Baro.x']].corr()
    return sns.heatmap(corr, annot=True)

In [6]:
def plot_sensor_data_by_label(X_train, y_train, label_cols,sensor_cols):
    for col in label_cols:
        for sensor in sensor_cols:
            mask_0 = y_train[col] == 0
            mask_1 = y_train[col] == 1
            print(mask_0.sum(), mask_1.sum())
            if  mask_0.sum()> 0:
                plt.title(f"{sensor} data over time for label {col}")
                filter_0_df = X_train[mask_0]
                plot_sensor_data_over_time(filter_0_df, sensor)
            if mask_1.sum() > 0:
                plt.title(f"{sensor} data over time for label {col}")
                filter_1_df = X_train[mask_1]
                plot_sensor_data_over_time(filter_1_df, sensor)

def plot_sensor_correlation_by_label(X_train, y_train, label_cols):
    for col in label_cols:
        if y_train[col].sum() == 0:
            print(f"Label [{col}] has no positive samples in the training set, skipping correlation plot.")
            continue
        filter_df = X_train[y_train[col] == 1]
        plt.title(f"Correlation of sensor data for label {col}")
        plot_sensor_correlation(filter_df)


In [7]:
sensor_cols = ["Acc.x","Acc.y","Acc.z","Gyro.x","Gyro.y","Gyro.z","Baro.x"]
def sliding_window_stats_overlapping(X_df, window_size=10000, step=5000, sensor_cols=None):

    if sensor_cols is None:
        sensor_cols = [c for c in X_df.columns if c != "time"]

    features = {
        "mean": [],
        "std": [],
        "min": [],
        "max": [],
        "median": []
    }

    for start in range(0, len(X_df) - window_size + 1, step):
        end = start + window_size
        window = X_df.iloc[start:end]

        mean_vals = window[sensor_cols].mean().values
        std_vals = window[sensor_cols].std().values
        min_vals = window[sensor_cols].min().values
        max_vals = window[sensor_cols].max().values
        median_vals = window[sensor_cols].median().values

        features["mean"].append(mean_vals)
        features["std"].append(std_vals)
        features["min"].append(min_vals)
        features["max"].append(max_vals)
        features["median"].append(median_vals)

    return features

In [8]:
def sliding_window_stats_non_overlapping(X_df, window_size=10000, sensor_cols=None):
    if sensor_cols is None:
        sensor_cols = [c for c in X_df.columns if c != "time"]

    features = {
        "mean": [],
        "std": [],
        "min": [],
        "max": [],
        "median": []
    }

    for start in range(0, len(X_df) - window_size + 1, window_size):
        end = start + window_size
        window = X_df.iloc[start:end]

        features["mean"].append(window[sensor_cols].mean().values)
        features["std"].append(window[sensor_cols].std().values)
        features["min"].append(window[sensor_cols].min().values)
        features["max"].append(window[sensor_cols].max().values)
        features["median"].append(window[sensor_cols].median().values)

    return features

In [9]:
def plot_all_features_sensor(features, sensor_name, sensor_cols, start=0, end=None):

    sensor_idx = sensor_cols.index(sensor_name)
    feature_names = list(features.keys())
    n_features = len(feature_names)

    fig, axes = plt.subplots(n_features, 1, figsize=(12, 3*n_features), sharex=True)

    for i, feature in enumerate(feature_names):
        data = np.array(features[feature])
        sensor_values = data[start:end, sensor_idx]

        axes[i].plot(range(start, start + len(sensor_values)), sensor_values)
        axes[i].set_title(f"{feature} of {sensor_name}")
        axes[i].set_ylabel(feature)
        axes[i].grid(True)

    axes[-1].set_xlabel("Window index")
    plt.tight_layout()
    plt.show()

In [10]:
overlapping_stats = sliding_window_stats_overlapping(X_train, window_size=10000, step=5000, sensor_cols=sensor_cols)
non_overlapping_stats = sliding_window_stats_non_overlapping(X_train, window_size=10000, sensor_cols=sensor_cols)

In [11]:
# for sensor in sensor_cols:
#     print("\n" + sensor)
#     plot_all_features_sensor(non_overlapping_stats, sensor, sensor_cols)

In [12]:
label_cols = y_train.columns.tolist()
sensor_cols = X_train.columns.tolist()[1:]  # Exclude 'time' column

# plot_sensor_data_by_label(X_train, y_train, label_cols, sensor_cols)